> **Student Edition (W02B)**  
> - Ejecuta los **DEMO** como guía.  
> - En **TU TURNO (1–4)** encontrarás `TODO`: debes escribir la consulta.  
> - Regla de oro: antes de un JOIN, verifica **grain** y **cardinalidad** (si no, duplicas filas).

# W03 – SQL esencial II (JOINs + CTEs) y cardinalidad práctica

## Conexión con DDIA
- **DDIA Cap. 2**: modelado + consultas; por qué las relaciones importan.
- Conexión práctica con **Cap. 3**: algunos joins se vuelven caros o peligrosos si la cardinalidad no está controlada.

## Prerrequisitos
- W02A (o dominar SELECT/WHERE/GROUP BY).
- Tener `raw_ps` disponible (el notebook puede descargar si falta).

## Objetivos
- Construir dimensiones simples (`dim_host`, `dim_discovery`).
- Usar `JOIN` (INNER/LEFT) y **demostrar** problemas de cardinalidad.
- Usar `CTE` (`WITH ...`) para estructurar consultas.
- Validar joins con conteos: evitar duplicación accidental.

## Checklist de evidencias
- [ ] Creaste `dim_host` y `dim_discovery`
- [ ] Mostraste un JOIN “malo” (duplica filas) + corrección
- [ ] 4 consultas del TU TURNO completas


In [1]:
# Setup común robusto para W03
import sys, subprocess
from pathlib import Path
import duckdb

# Fijar manualmente la raíz del proyecto
PROJECT_ROOT = Path(r"C:\Users\USUARIO WINDOWS\CODIGOS\PAZ CD").resolve()

DB_PATH = PROJECT_ROOT / "data" / "exoplanets.duckdb"
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

con = duckdb.connect(str(DB_PATH))

raw_csv = PROJECT_ROOT / "data" / "raw" / "pscomppars.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("raw_csv:", raw_csv)
print("Existe raw_csv:", raw_csv.exists())

if not raw_csv.exists():
    raise FileNotFoundError(
        f"No encontré el archivo Raw en:\n{raw_csv}\n\n"
        "Como ya lo descargaste, revisa que esté exactamente en data/raw/pscomppars.csv"
    )

def sql_quote(s: str) -> str:
    return "'" + s.replace("'", "''") + "'"

raw_csv_abs = raw_csv.resolve()

con.execute(
    f"""
    CREATE OR REPLACE VIEW raw_ps AS
    SELECT * FROM read_csv_auto({sql_quote(raw_csv_abs.as_posix())})
    """
)

con.sql("SELECT count(*) AS n_rows FROM raw_ps").show()
con.sql("DESCRIBE raw_ps").show()

PROJECT_ROOT: C:\Users\USUARIO WINDOWS\CODIGOS\PAZ CD
raw_csv: C:\Users\USUARIO WINDOWS\CODIGOS\PAZ CD\data\raw\pscomppars.csv
Existe raw_csv: True
┌────────┐
│ n_rows │
│ int64  │
├────────┤
│   6291 │
└────────┘

┌─────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name   │ column_type │  null   │   key   │ default │  extra  │
│     varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ pl_name         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ hostname        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ discoverymethod │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ disc_year       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ sy_snum         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ sy_pnum         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ sy_dist         │ DOUBLE      │ YES     │ N

In [2]:
# (Opcional) Ver columnas disponibles en raw_ps (útil para depurar)
con.sql("DESCRIBE raw_ps").show()

┌─────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name   │ column_type │  null   │   key   │ default │  extra  │
│     varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ pl_name         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ hostname        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ discoverymethod │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ disc_year       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ sy_snum         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ sy_pnum         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ sy_dist         │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ ra              │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ dec             │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ pl_orbper       │ DOUBLE      │ YES 

## DEMO (docente)


In [3]:
# DEMO 1: dimensión de hosts (1 fila por hostname) SOLO con SQL básico
# Clave: hostname
#
# Importante:
# - DISTINCT sobre (hostname, ra, ...) NO garantiza 1 fila por hostname.
# - Para forzar 1 fila por hostname SIN window functions, agregamos por hostname.
# - MAX(...) es una forma simple de "escoger un valor" y además ignora NULLs.
#
# Nota pedagógica: ejemplo didáctico. Más adelante veremos políticas de resolución más robustas.

con.execute("""
CREATE OR REPLACE TABLE dim_host_ra AS
SELECT
  hostname,
  MAX(ra) AS ra
FROM raw_ps
WHERE hostname IS NOT NULL
GROUP BY hostname
""")

# Validación rápida: en una dimensión correcta, n_rows == n_keys
con.sql("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT hostname) AS n_keys FROM dim_host_ra").show()

┌────────┬────────┐
│ n_rows │ n_keys │
│ int64  │ int64  │
├────────┼────────┤
│   4709 │   4709 │
└────────┴────────┘



In [4]:
con.execute("""
CREATE OR REPLACE TABLE dim_discovery AS
SELECT DISTINCT discoverymethod, disc_year
FROM raw_ps
""")
con.execute("SELECT count(*) FROM dim_discovery").fetchall()


[(142,)]

In [5]:
# DEMO 2: fact (grano = 1 fila por planeta)
con.execute("""
CREATE OR REPLACE TABLE fact_planet_raw AS
SELECT
  pl_name,
  hostname,
  discoverymethod,
  disc_year,
  pl_orbper,
  pl_rade,
  pl_bmasse,
  pl_eqt
FROM raw_ps
WHERE pl_name IS NOT NULL
""")
con.sql("SELECT count(*) FROM fact_planet_raw").show()


┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         6291 │
└──────────────┘



### DEMO 3: JOIN correcto (many-to-one)
`fact_planet_raw` → `dim_host` debería ser muchos-a-uno. Si `dim_host` tiene hostname único, **no debe multiplicar filas**.


In [6]:
n_fact = con.execute("SELECT count(*) FROM fact_planet_raw").fetchone()[0]
n_join = con.execute("""
SELECT count(*)
FROM fact_planet_raw f
JOIN dim_host_ra h
  ON f.hostname = h.hostname
""").fetchone()[0]
n_fact, n_join


(6291, 6291)

### DEMO 4: JOIN “malo” (duplica filas)
Error común: unirse a una tabla que **no es dimensión** (llave no única). Fabricamos una “dimensión mala” a propósito.


In [7]:
# DEMO 3: una "dimensión" MAL construida (violando 1 fila por hostname)
# Aquí NO deduplicamos: tendrá múltiples filas por hostname.
con.execute("""
CREATE OR REPLACE TABLE dim_host_bad AS
SELECT hostname, ra
FROM raw_ps
WHERE hostname IS NOT NULL
""")

# Evidencia sin HAVING (solo CTE + WHERE)
con.sql("""
WITH c AS (
  SELECT hostname, COUNT(*) AS cnt
  FROM dim_host_bad
  GROUP BY hostname
)
SELECT * FROM c
WHERE cnt > 1
ORDER BY cnt DESC
LIMIT 10
""").show()

┌────────────┬───────┐
│  hostname  │  cnt  │
│  varchar   │ int64 │
├────────────┼───────┤
│ KOI-351    │     8 │
│ TRAPPIST-1 │     7 │
│ Kepler-20  │     6 │
│ HD 219134  │     6 │
│ TOI-178    │     6 │
│ HD 191939  │     6 │
│ HD 10180   │     6 │
│ Kepler-80  │     6 │
│ HD 110067  │     6 │
│ K2-138     │     6 │
├────────────┴───────┤
│ 10 rows  2 columns │
└────────────────────┘



In [8]:
n_join_bad = con.execute("""
SELECT count(*)
FROM fact_planet_raw f
JOIN dim_host_bad h
  ON f.hostname = h.hostname
""").fetchone()[0]

n_fact, n_join_bad


(6291, 11045)

### DEMO 5: arreglar JOIN malo con CTE (deduplicación)


In [9]:
n_join_fixed = con.execute("""
WITH dim_host_fixed AS (
  SELECT DISTINCT hostname
  FROM dim_host_bad
)
SELECT count(*)
FROM fact_planet_raw f
JOIN dim_host_fixed h
  ON f.hostname = h.hostname
""").fetchone()[0]
n_fact, n_join_fixed


(6291, 6291)

## TU TURNO (práctica guiada)


### 1) LEFT JOIN y no-match: ¿cuántas filas quedan sin match en dim_host?

In [10]:
# TODO (1) LEFT JOIN y no-match
# Objetivo: ¿cuántas filas de fact_planet quedan SIN match en dim_host?
# Pistas:
# - Usa LEFT JOIN fact_planet f con dim_host h ON f.hostname = h.hostname
# - Cuenta las filas donde h.hostname IS NULL
query = """
SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN h.hostname IS NULL THEN 1 ELSE 0 END) AS no_match
FROM fact_planet_raw f
LEFT JOIN dim_host_ra h
  ON f.hostname = h.hostname
"""

con.sql(query).show()

┌───────┬──────────┐
│ total │ no_match │
│ int64 │  int128  │
├───────┼──────────┤
│  6291 │        0 │
└───────┴──────────┘



### 2) CTE + ranking: por año, método #1 (más planetas)

In [11]:
# TODO (2) CTE + ranking
# Objetivo: por cada disc_year, obtener el discoverymethod #1 (más planetas) y su conteo
# Pistas:
# - Agrupa por disc_year, discoverymethod y cuenta
# - Usa una ventana: ROW_NUMBER() OVER(PARTITION BY disc_year ORDER BY n DESC)
# - Filtra rn = 1
query = """
WITH counts_by_year_method AS (
  SELECT
    disc_year,
    discoverymethod,
    COUNT(*) AS n_planets
  FROM fact_planet_raw
  WHERE disc_year IS NOT NULL
    AND discoverymethod IS NOT NULL
  GROUP BY disc_year, discoverymethod
),
ranked AS (
  SELECT
    disc_year,
    discoverymethod,
    n_planets,
    ROW_NUMBER() OVER (
      PARTITION BY disc_year
      ORDER BY n_planets DESC, discoverymethod ASC
    ) AS rn
  FROM counts_by_year_method
)
SELECT
  disc_year,
  discoverymethod,
  n_planets
FROM ranked
WHERE rn = 1
ORDER BY disc_year ASC
"""

con.execute(query).fetchall()

[(1992, 'Pulsar Timing', 2),
 (1994, 'Pulsar Timing', 1),
 (1995, 'Radial Velocity', 1),
 (1996, 'Radial Velocity', 6),
 (1997, 'Radial Velocity', 1),
 (1998, 'Radial Velocity', 6),
 (1999, 'Radial Velocity', 13),
 (2000, 'Radial Velocity', 16),
 (2001, 'Radial Velocity', 12),
 (2002, 'Radial Velocity', 28),
 (2003, 'Radial Velocity', 21),
 (2004, 'Radial Velocity', 18),
 (2005, 'Radial Velocity', 33),
 (2006, 'Radial Velocity', 21),
 (2007, 'Radial Velocity', 34),
 (2008, 'Radial Velocity', 36),
 (2009, 'Radial Velocity', 70),
 (2010, 'Transit', 47),
 (2011, 'Transit', 79),
 (2012, 'Transit', 93),
 (2013, 'Transit', 79),
 (2014, 'Transit', 798),
 (2015, 'Transit', 99),
 (2016, 'Transit', 1432),
 (2017, 'Transit', 87),
 (2018, 'Transit', 244),
 (2019, 'Transit', 107),
 (2020, 'Transit', 164),
 (2021, 'Transit', 457),
 (2022, 'Transit', 191),
 (2023, 'Transit', 224),
 (2024, 'Transit', 187),
 (2025, 'Transit', 144),
 (2026, 'Transit', 157)]

### 3) Validación de cardinalidad: ¿hay duplicados en (discoverymethod, disc_year) en dim_discovery?

In [12]:
# TODO (3) Validación de cardinalidad
# Objetivo: detectar si hay duplicados en la clave (discoverymethod, disc_year) dentro de dim_discovery
# Pistas:
# - GROUP BY discoverymethod, disc_year
# - HAVING COUNT(*) > 1
query = """
SELECT
  discoverymethod,
  disc_year,
  COUNT(*) AS n_rows
FROM dim_discovery
GROUP BY discoverymethod, disc_year
HAVING COUNT(*) > 1
ORDER BY n_rows DESC, discoverymethod, disc_year
"""

con.execute(query).fetchall()

[]

### 4) JOIN + agregación: promedio de RA del host por método

In [13]:
# TODO (4) JOIN + agregación
# Objetivo: para cada discoverymethod, contar planetas y calcular el promedio de RA del host.
# Nota: dim_host_ra solo tiene (hostname, ra). Por eso usamos ra (no st_teff).
# Pistas:
# - JOIN fact_planet_raw (f) con dim_host_ra (h) por hostname
# - Filtra discoverymethod y ra no nulos
# - GROUP BY discoverymethod
query = """
SELECT
  f.discoverymethod,
  COUNT(*) AS n_planets,
  ROUND(AVG(h.ra), 2) AS avg_ra
FROM fact_planet_raw f
JOIN dim_host_ra h
  ON f.hostname = h.hostname
WHERE f.discoverymethod IS NOT NULL
  AND h.ra IS NOT NULL
GROUP BY f.discoverymethod
ORDER BY n_planets DESC
"""

con.sql(query).show()

┌───────────────────────────────┬───────────┬────────┐
│        discoverymethod        │ n_planets │ avg_ra │
│            varchar            │   int64   │ double │
├───────────────────────────────┼───────────┼────────┤
│ Transit                       │      4651 │ 243.76 │
│ Radial Velocity               │      1181 │ 175.86 │
│ Microlensing                  │       278 │ 267.68 │
│ Imaging                       │        97 │  186.2 │
│ Transit Timing Variations     │        41 │ 245.14 │
│ Eclipse Timing Variations     │        17 │ 223.46 │
│ Orbital Brightness Modulation │         9 │ 292.91 │
│ Pulsar Timing                 │         8 │ 218.74 │
│ Astrometry                    │         6 │ 235.42 │
│ Pulsation Timing Variations   │         2 │ 315.24 │
│ Disk Kinematics               │         1 │ 167.01 │
├───────────────────────────────┴───────────┴────────┤
│ 11 rows                                  3 columns │
└────────────────────────────────────────────────────┘



## DEMO (capstone, opcional) — CTE + Window Functions (estilo “heroes”)

> **Esta sección es opcional y NO entra en el entregable obligatorio.**  
> La idea es cerrar la teoría con un ejemplo “potente” como el que vimos con `heroes/team`:
> - 1er CTE: limpiamos/filtramos filas (como `heroes_clean`)
> - 2do CTE: calculamos métricas por grupo con **window functions** y rankeamos (como `ranked`)
> - SELECT final: nos quedamos con **la fila `rn=1`** por grupo (el “top 1” por partición)

**Pregunta análoga (en exoplanetas):**  
Para cada `hostname` (sistema), encontrar el **planeta con mayor radio** (`pl_rade`) y además reportar:
- `avg_radius_in_system` (promedio de radios en el sistema)
- `planets_in_system` (cuántos planetas tiene el sistema)
- un atributo del host (`ra`, si existe)

In [ ]:
# CTE + Window Functions (capstone)
query = r'''
WITH planets_clean AS (
  SELECT
    f.pl_name,
    f.hostname,
    f.pl_rade,
    h.ra
  FROM fact_planet_raw f
  LEFT JOIN dim_host_ra h
    ON h.hostname = f.hostname
  WHERE f.hostname IS NOT NULL
    AND f.pl_rade IS NOT NULL
    AND f.pl_rade BETWEEN 0 AND 30   -- filtro tipo "edad entre 0 y 120"
),
ranked AS (
  SELECT
    hostname,
    ra,
    pl_name,
    pl_rade,
    AVG(pl_rade) OVER (PARTITION BY hostname) AS avg_radius_in_system,
    COUNT(*)     OVER (PARTITION BY hostname) AS planets_in_system,
    ROW_NUMBER() OVER (
      PARTITION BY hostname
      ORDER BY pl_rade DESC, pl_name ASC
    ) AS rn
  FROM planets_clean
)
SELECT
  hostname,
  ra,
  pl_name AS largest_planet,
  pl_rade AS largest_radius,
  avg_radius_in_system,
  planets_in_system
FROM ranked
WHERE rn = 1
ORDER BY planets_in_system DESC, hostname
LIMIT 50;
'''
con.sql(query).show()

In [72]:
try:
    con.close()
    print("DuckDB connection closed.")
except NameError:
    print("No connection named 'con' in this notebook.")

DuckDB connection closed.


## Para entregar (W03) 

### En clase
1) `docs/w03_sql_practice.md` con el análisis de `dim_host_bad`:
   - Los 4 conteos: `n_fact`, `n_join_good`, `n_join_bad`, `n_join_fixed` + 2–3 líneas explicando qué pasó.
   - Respuestas a **TODO 1–4** (cada una: SQL + output pegado).

2) `docs/decisions_log.md`: 1 entrada corta:
   - “Cómo validé cardinalidad antes de un JOIN” + evidencia (una query de duplicados o un conteo).

> **Nota:** La sección **DEMO capstone (CTE + Window)** es **solo demostración** (no se entrega).

### Tarea (para la próxima clase)
1) `docs/w03_join_case.md`: **1 caso real de JOIN malo**
   - evidencia con conteos antes/después
   - diagnóstico (qué clave falló)
   - fix simple (por ejemplo: dedupe con `GROUP BY`/`DISTINCT` o pre-agregación)

2) 2 consultas extra (en `docs/w03_sql_practice.md`):
   - 1 que incluya `JOIN`
   - 1 que incluya `CTE`

SOLUCION

In [13]:
con.execute("""
CREATE OR REPLACE TABLE dim_host_bad AS
SELECT
  hostname,
  ra
FROM raw_ps
WHERE hostname IS NOT NULL
""")

con.sql("""
SELECT COUNT(*) AS n_rows,
       COUNT(DISTINCT hostname) AS n_keys
FROM dim_host_bad
""").show()

┌────────┬────────┐
│ n_rows │ n_keys │
│ int64  │ int64  │
├────────┼────────┤
│   6291 │   4709 │
└────────┴────────┘



In [14]:
con.sql("""
WITH c AS (
  SELECT hostname, COUNT(*) AS cnt
  FROM dim_host_bad
  GROUP BY hostname
)
SELECT *
FROM c
WHERE cnt > 1
ORDER BY cnt DESC
LIMIT 10
""").show()

┌────────────┬───────┐
│  hostname  │  cnt  │
│  varchar   │ int64 │
├────────────┼───────┤
│ KOI-351    │     8 │
│ TRAPPIST-1 │     7 │
│ HD 110067  │     6 │
│ TOI-178    │     6 │
│ Kepler-80  │     6 │
│ Kepler-11  │     6 │
│ HD 10180   │     6 │
│ Kepler-20  │     6 │
│ HD 34445   │     6 │
│ HIP 41378  │     6 │
├────────────┴───────┤
│ 10 rows  2 columns │
└────────────────────┘



In [16]:
# REPARAR TABLAS NECESARIAS PARA W03

# 1) Tabla de hechos: una fila por planeta
con.execute("""
CREATE OR REPLACE TABLE fact_planet_raw AS
SELECT
  pl_name,
  hostname,
  discoverymethod,
  disc_year,
  pl_orbper,
  pl_rade,
  pl_bmasse,
  pl_eqt
FROM raw_ps
WHERE pl_name IS NOT NULL
""")

# 2) Dimensión buena: una fila por hostname
con.execute("""
CREATE OR REPLACE TABLE dim_host_ra AS
SELECT
  hostname,
  MAX(ra) AS ra
FROM raw_ps
WHERE hostname IS NOT NULL
GROUP BY hostname
""")

# 3) Dimensión mala: varias filas por hostname, creada a propósito
con.execute("""
CREATE OR REPLACE TABLE dim_host_bad AS
SELECT
  hostname,
  ra
FROM raw_ps
WHERE hostname IS NOT NULL
""")

# 4) Dimensión discovery
con.execute("""
CREATE OR REPLACE TABLE dim_discovery AS
SELECT DISTINCT
  discoverymethod,
  disc_year
FROM raw_ps
""")

# Verificación rápida
con.sql("""
SELECT 'fact_planet_raw' AS tabla, COUNT(*) AS n_rows FROM fact_planet_raw
UNION ALL
SELECT 'dim_host_ra' AS tabla, COUNT(*) AS n_rows FROM dim_host_ra
UNION ALL
SELECT 'dim_host_bad' AS tabla, COUNT(*) AS n_rows FROM dim_host_bad
UNION ALL
SELECT 'dim_discovery' AS tabla, COUNT(*) AS n_rows FROM dim_discovery
""").show()

┌─────────────────┬────────┐
│      tabla      │ n_rows │
│     varchar     │ int64  │
├─────────────────┼────────┤
│ fact_planet_raw │   6291 │
│ dim_host_ra     │   4709 │
│ dim_host_bad    │   6291 │
│ dim_discovery   │    142 │
└─────────────────┴────────┘



In [17]:
n_fact = con.execute("""
SELECT COUNT(*)
FROM fact_planet_raw
""").fetchone()[0]

n_join_bad = con.execute("""
SELECT COUNT(*)
FROM fact_planet_raw f
JOIN dim_host_bad h
  ON f.hostname = h.hostname
""").fetchone()[0]

multiplicacion = n_join_bad - n_fact

print("n_fact:", n_fact)
print("n_join_bad:", n_join_bad)
print("Filas adicionales por JOIN malo:", multiplicacion)

n_fact: 6291
n_join_bad: 11045
Filas adicionales por JOIN malo: 4754


In [18]:
multiplicacion = n_join_bad - n_fact
print("Filas adicionales por JOIN malo:", multiplicacion)

Filas adicionales por JOIN malo: 4754


CORRECCION JOIN MALO

In [19]:
con.execute("""
CREATE OR REPLACE TABLE dim_host_fixed AS
SELECT
  hostname,
  MAX(ra) AS ra
FROM dim_host_bad
GROUP BY hostname
""")

con.sql("""
SELECT COUNT(*) AS n_rows,
       COUNT(DISTINCT hostname) AS n_keys
FROM dim_host_fixed
""").show()

┌────────┬────────┐
│ n_rows │ n_keys │
│ int64  │ int64  │
├────────┼────────┤
│   4709 │   4709 │
└────────┴────────┘



TODO 1 — LEFT JOIN y no-match

In [20]:
query = """
SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN h.hostname IS NULL THEN 1 ELSE 0 END) AS no_match
FROM fact_planet_raw f
LEFT JOIN dim_host_ra h
  ON f.hostname = h.hostname
"""
con.sql(query).show()

┌───────┬──────────┐
│ total │ no_match │
│ int64 │  int128  │
├───────┼──────────┤
│  6291 │        0 │
└───────┴──────────┘



TODO 2 — CTE + ranking: método principal por año

In [21]:
query = """
WITH counts AS (
  SELECT
    disc_year,
    discoverymethod,
    COUNT(*) AS n
  FROM fact_planet_raw
  WHERE disc_year IS NOT NULL
    AND discoverymethod IS NOT NULL
  GROUP BY disc_year, discoverymethod
),
ranked AS (
  SELECT
    disc_year,
    discoverymethod,
    n,
    ROW_NUMBER() OVER (
      PARTITION BY disc_year
      ORDER BY n DESC
    ) AS rn
  FROM counts
)
SELECT
  disc_year,
  discoverymethod,
  n
FROM ranked
WHERE rn = 1
ORDER BY disc_year
"""
con.execute(query).fetchall()

[(1992, 'Pulsar Timing', 2),
 (1994, 'Pulsar Timing', 1),
 (1995, 'Radial Velocity', 1),
 (1996, 'Radial Velocity', 6),
 (1997, 'Radial Velocity', 1),
 (1998, 'Radial Velocity', 6),
 (1999, 'Radial Velocity', 13),
 (2000, 'Radial Velocity', 16),
 (2001, 'Radial Velocity', 12),
 (2002, 'Radial Velocity', 28),
 (2003, 'Radial Velocity', 21),
 (2004, 'Radial Velocity', 18),
 (2005, 'Radial Velocity', 33),
 (2006, 'Radial Velocity', 21),
 (2007, 'Radial Velocity', 34),
 (2008, 'Radial Velocity', 36),
 (2009, 'Radial Velocity', 70),
 (2010, 'Transit', 47),
 (2011, 'Transit', 79),
 (2012, 'Transit', 93),
 (2013, 'Transit', 79),
 (2014, 'Transit', 798),
 (2015, 'Transit', 99),
 (2016, 'Transit', 1432),
 (2017, 'Transit', 87),
 (2018, 'Transit', 244),
 (2019, 'Transit', 107),
 (2020, 'Transit', 164),
 (2021, 'Transit', 457),
 (2022, 'Transit', 191),
 (2023, 'Transit', 224),
 (2024, 'Transit', 187),
 (2025, 'Transit', 144),
 (2026, 'Transit', 157)]

TODO 3 — Validación de cardinalidad en dim_discovery

In [23]:
query = """
SELECT
  discoverymethod,
  disc_year,
  COUNT(*) AS cnt
FROM dim_discovery
GROUP BY discoverymethod, disc_year
HAVING COUNT(*) > 1
ORDER BY cnt DESC
"""
con.execute(query).fetchall()

[]

TODO 4

In [24]:
query = """
SELECT
  f.discoverymethod,
  COUNT(*) AS n_planets,
  ROUND(AVG(h.ra), 2) AS avg_ra
FROM fact_planet_raw f
JOIN dim_host_ra h
  ON f.hostname = h.hostname
WHERE f.discoverymethod IS NOT NULL
  AND h.ra IS NOT NULL
GROUP BY f.discoverymethod
ORDER BY n_planets DESC
"""
con.sql(query).show()

┌───────────────────────────────┬───────────┬────────┐
│        discoverymethod        │ n_planets │ avg_ra │
│            varchar            │   int64   │ double │
├───────────────────────────────┼───────────┼────────┤
│ Transit                       │      4651 │ 243.76 │
│ Radial Velocity               │      1181 │ 175.86 │
│ Microlensing                  │       278 │ 267.68 │
│ Imaging                       │        97 │  186.2 │
│ Transit Timing Variations     │        41 │ 245.14 │
│ Eclipse Timing Variations     │        17 │ 223.46 │
│ Orbital Brightness Modulation │         9 │ 292.91 │
│ Pulsar Timing                 │         8 │ 218.74 │
│ Astrometry                    │         6 │ 235.42 │
│ Pulsation Timing Variations   │         2 │ 315.24 │
│ Disk Kinematics               │         1 │ 167.01 │
├───────────────────────────────┴───────────┴────────┤
│ 11 rows                                  3 columns │
└────────────────────────────────────────────────────┘



CONSULTA EXTRA

In [25]:
query = """
SELECT
  f.pl_name,
  f.pl_bmasse,
  h.ra
FROM fact_planet_raw f
JOIN dim_host_ra h
  ON f.hostname = h.hostname
WHERE f.pl_bmasse IS NOT NULL
ORDER BY f.pl_bmasse DESC
LIMIT 10
"""
con.sql(query).show()

┌────────────────────────────┬───────────────┬─────────────┐
│          pl_name           │   pl_bmasse   │     ra      │
│          varchar           │    double     │   double    │
├────────────────────────────┼───────────────┼─────────────┤
│ 2MASS J22501512+2325342 b  │    9534.85221 │ 342.5634097 │
│ CD-35 2722 b               │  9375.9380065 │  92.3300147 │
│ Luhman 16 b                │  9344.1551658 │ 162.3282594 │
│ HD 188641 b                │ 9333.03117155 │ 299.4048303 │
│ KMT-2018-BLG-0885L b       │   9217.023803 │ 268.9663333 │
│ DENIS-P J082303.1-491201 b │       9057.77 │ 125.7620668 │
│ HD 26161 b                 │ 9045.39646322 │  62.4122255 │
│ HD 6860 b                  │ 8981.83078182 │   17.432979 │
│ 2MASS J11011926-7732383 b  │   8899.195396 │ 165.3295388 │
│ TOI-5422 b                 │   8899.195396 │  86.8436173 │
├────────────────────────────┴───────────────┴─────────────┤
│ 10 rows                                        3 columns │
└───────────────────────

In [26]:
query = """
WITH metodos AS (
  SELECT
    disc_year,
    discoverymethod,
    COUNT(*) AS n
  FROM fact_planet_raw
  WHERE disc_year IS NOT NULL
    AND discoverymethod IN ('Transit', 'Radial Velocity')
  GROUP BY disc_year, discoverymethod
),
comparacion AS (
  SELECT
    disc_year,
    MAX(CASE WHEN discoverymethod = 'Transit' THEN n END) AS transit,
    MAX(CASE WHEN discoverymethod = 'Radial Velocity' THEN n END) AS radial_velocity
  FROM metodos
  GROUP BY disc_year
)
SELECT *
FROM comparacion
WHERE transit > radial_velocity
ORDER BY disc_year
"""
con.sql(query).show()

┌───────────┬─────────┬─────────────────┐
│ disc_year │ transit │ radial_velocity │
│   int64   │  int64  │      int64      │
├───────────┼─────────┼─────────────────┤
│      2010 │      47 │              40 │
│      2011 │      79 │              42 │
│      2012 │      93 │              35 │
│      2013 │      79 │              34 │
│      2014 │     798 │              48 │
│      2015 │      99 │              46 │
│      2016 │    1432 │              58 │
│      2017 │      87 │              48 │
│      2018 │     244 │              38 │
│      2019 │     107 │              65 │
│      2020 │     164 │              46 │
│      2021 │     457 │              77 │
│      2022 │     191 │             117 │
│      2023 │     224 │              56 │
│      2024 │     187 │              29 │
│      2025 │     144 │              61 │
│      2026 │     157 │              25 │
├───────────┴─────────┴─────────────────┤
│ 17 rows                     3 columns │
└─────────────────────────────────